  Implementation for simulating unsafe situations in SUMO with detailed configurations and collision-inducing mechanisms

1. Dangerous Junction Configuration (dangerous_junction.add.xml)

In [ ]:
<additional>
    <junction id="high_risk_junc" type="priority">
        <param key="jmIgnoreFoeProb" value="0.4"/>
        <param key="jmDriveAfterRedTime" value="3.0"/>
        <param key="jmTimegapMinor" value="0.7"/>
        <param key="collision.check-junctions" value="true"/>
    </junction>

    <vType id="aggressive_car" accel="3.5" decel="3.0" sigma="0.8"
           emergencyDecel="4.5" tau="0.5" speedFactor="1.3"/>
</additional>


2. Collision Forcing Script (unsafe_control.py)

In [ ]:
import traci
import random

def create_conflict():
    # Force opposing vehicles into collision course
    traci.vehicle.setSpeed("veh1", 25)
    traci.vehicle.setSpeedMode("veh1", 0)  # Disable safety checks[2]
    traci.vehicle.setLaneChangeMode("veh1", 512)  # Force immediate lane change[2]

    traci.vehicle.setSpeed("veh2", 30)
    traci.vehicle.setRouteID("veh2", "conflict_route")

traci.start(["sumo-gui", "-c", "unsafe.net.xml", "--collision-output", "collisions.xml"])

try:
    while traci.simulation.getMinExpectedNumber() > 0:
        traci.simulationStep()

        # Randomly trigger dangerous maneuvers
        if random.random() < 0.1:  # 10% chance per step
            veh_list = traci.vehicle.getIDList()
            if len(veh_list) >= 2:
                create_conflict()

        # Override following parameters dynamically
        for veh_id in traci.vehicle.getIDList():
            traci.vehicle.setParameter(veh_id, "carFollowModel.emergencyDecel", "3.0")
            traci.vehicle.setParameter(veh_id, "carFollowModel.apparentDecel", "2.5")

except traci.FatalTraCIError:
    print("Simulation terminated with collisions")
finally:
    traci.close()


3. Safety Measure Configuration (ssm_config.add.xml)

In [ ]:
<configuration>
    <output>
        <safety-output value="TTC"/>          <!-- Time-to-Collision -->
        <safety-output value="DRAC"/>         <!-- Deceleration Rate -->
        <safety-output value="PET"/>          <!-- Post-Encroachment Time -->
        <safety-output value="collisions"/>   <!-- Physical collisions[4] -->
    </output>
    <processing>
        <collision.action value="warn"/>      <!-- Log but continue simulation -->
        <collision.mingap-factor value="0"/>  <!-- Detect physical overlaps[2] -->
    </processing>
</configuration>


4. Post-Simulation Analysis

In [ ]:
from sumolib.output import parse
import pandas as pd

def analyze_collisions():
    collision_data = []
    for collision in parse("collisions.xml", "collision"):
        collision_data.append({
            "time": float(collision.time),
            "type": collision.type,
            "collider": collision.collider,
            "victim": collision.victim,
            "speed_diff": float(collision.colliderSpeed) - float(collision.victimSpeed),
            "pos": float(collision.pos)
        })

    df = pd.DataFrame(collision_data)
    print(f"Total collisions: {len(df)}")
    print(f"Most common collision type: {df['type'].mode()[0]}")
    return df

collision_df = analyze_collisions()


jmIgnoreFoeProb	0.2-0.6	Right-of-way violation probability
speedFactor	1.2-1.8	Speeding beyond limit
tau	0.3-0.8	Reduced reaction time
emergencyDecel	2.0-3.5 m/s²	Limited emergency braking capability
laneChangeMode	512	Disable safety checks for lane changes

Critical SSM Thresholds

In [ ]:
# Real-time safety monitoring
SAFE_TTC = 2.5  # seconds
CRITICAL_DRAC = 3.0  # m/s²

def check_safety():
    ttc = traci.vehicle.getParameter(veh_id, "device.ssm.TTC")
    if float(ttc) < SAFE_TTC:
        trigger_emergency_response()


This implementation creates controlled unsafe scenarios by:

Configuring aggressive driver behavior parameters

Forcing collisions through TraCI overrides

Monitoring surrogate safety measures

Analyzing collision patterns post-simulation

To visualize safety metrics in real-time:

In [ ]:
python tools/visualization/plot_ssm.py -i ssm.xml --thresholds TTC=2.5,DRAC=3.0
